# Test du wrapper MME_Global et méthodes générales


In [1]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

['/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python310.zip', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/lib-dynload', '', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages', '/home/sagemaker-user/gbm_hackathon']


In [2]:
%load_ext autoreload
%autoreload 2
import os
import json
import pickle as pkl
from gbmhackathon.utils import global_wrapper

Load la config et effectue quelques modifications (j'arrive pas à effectuer les modifs dans sagemaker c'est relou)

In [13]:
path_config="../gbmhackathon/utils/config.json"
with open(path_config, 'r') as f:
    config= json.load(f)

print(config.keys())

output_dir = "/home/sagemaker-user/results"
os.makedirs(output_dir, exist_ok=True)
config['global_settings']["output_dir"]="/home/sagemaker-user/results" 
config["MME_Model"]["training"]["epochs"]=10
del config["MME_Model"]["modalities_data"]["modalities"]["spatial"] ## Marche pas à cause de timothée
del config["MME_Model"]["modalities_data"]["modalities"]["wes"] ## Marche pas à cause de timothée

print(config)
config["Experiments"]["Multiple_run_experiment"]["params"]["n_run"]=5

dict_keys(['global_settings', 'MME_Model', 'Experiments'])
{'global_settings': {'device': 'cpu', 'output_dir': '/home/sagemaker-user/results'}, 'MME_Model': {'modalities_data': {'modalities': {'hne': 'embeddings_HnE_OptimusH0.pkl', 'clinical': '2025-03-30_14-23_clinical_emb_V1.pkl'}, 'pkl_storage_folder': 'embedding_V1', 'missing_mods': 's3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/missing_mod_per_samples.pkl'}, 'training': {'batch_size': 16, 'epochs': 10, 'InfoNCE_Loss': {'temperature': 0.05, 'similarity': 'nt-xent', 'alpha': 0.1, 'bound': -10, 'beta': 0.2, 'nce_eps': 1e-08, 'reg_eps': 1e-08, 'use_all_positives': False}, 'Optimizer': {'lr': 0.001}}, 'architecture': {'MME': {'hne_cfg': {'net_type': 'mlp', 'net_config': {'layers': [1536, 512, 64], 'dropout': 0.65, 'act_fn': 'torch.nn.ReLU', 'norm_layer': 'torch.nn.LayerNorm'}}, 'spatial_cfg': {'net_type': 'attention', 'net_config': {'dim': 1790, 'depth': 4, 'num_heads': 8, 'mlp_ratio': 4.0, 'qkv_bias': True,

On instancie la classe, avec la config 


In [14]:
wahou=global_wrapper.MME_Global(config)

Using device: cpu
hne torch.Size([16, 1536])
clinical torch.Size([16, 12])
Using device: cpu
No potential residual connections found
Using device: cpu
No potential residual connections found


pour l'entraîner : 

In [15]:
wahou.fit_mme()

/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:26: ParameterNotFoundWarning: Parameter beta not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate'].
  warnings.warn(
/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:26: ParameterNotFoundWarning: Parameter nce_eps not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate'].
  warnings.warn(
/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:26: ParameterNotFoundWarning: Parameter reg_eps not found in init arguments: ['self', 'modalities', 'patient_map', 'temperature', 'similarity', 'use_all_positives', 'alpha', 'eps', 'bound', 'slope', 'rate'].
  warnings.warn(


hne
clinical
tensor(1.1328, grad_fn=<DivBackward0>)
hne
clinical
tensor(0.7162, grad_fn=<DivBackward0>)
hne
clinical
tensor(1.9126, grad_fn=<DivBackward0>)
hne
clinical
tensor(0.9678, grad_fn=<DivBackward0>)
hne
clinical
tensor(0.8769, grad_fn=<DivBackward0>)
hne
clinical
tensor(-0.2574, grad_fn=<DivBackward0>)
hne
clinical
tensor(-1.1396, grad_fn=<DivBackward0>)
hne
clinical
tensor(-12.9293, grad_fn=<DivBackward0>)
EPOCH 0 LOSS: 4.8414
hne
clinical
tensor(0.8571, grad_fn=<DivBackward0>)
hne
clinical
tensor(-1.5165, grad_fn=<DivBackward0>)
hne
clinical
tensor(2.3069, grad_fn=<DivBackward0>)
hne
clinical
tensor(-2.0152, grad_fn=<DivBackward0>)
hne
clinical
tensor(1.0687, grad_fn=<DivBackward0>)
hne
clinical
tensor(2.1236, grad_fn=<DivBackward0>)
hne
clinical
tensor(1.6817, grad_fn=<DivBackward0>)
hne
clinical
tensor(-9.4376, grad_fn=<DivBackward0>)
EPOCH 1 LOSS: 4.8988
hne
clinical
tensor(-0.6014, grad_fn=<DivBackward0>)
hne
clinical
tensor(-2.6134, grad_fn=<DivBackward0>)
hne
clinical


Pour sauvegarder : attention, tout est sauvegardé dans le dossier spécifié en config (voir ligne : config['global_settings']["output_dir"]="/home/sagemaker-user/results"  au dessu)
Cela va sauvegarder une instance de la classe MME_Global, et son modèle dans un fichier séparé
--> De la sorte, on peut tout récupérer et savoir ce qui a été fait 

In [ ]:
wahou.save()


si on veut récupérer l'ensemble à postériorie : 

In [ ]:
with open("path/to/MME_global.pkl", "rb") as f:
    old_MME = pkl.load(f)
print("model : ", old_MME.mme)

In [ ]:
old_MME.reload_model()
print("now, model : ",old_MME.mme)

